In [2]:
import os
import sys
import re
import h5py
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
# Add the parent directory (project root) to Python path
project_root = os.path.abspath('..')  # Go up one level from current notebook
if project_root not in sys.path:
    sys.path.insert(0, project_root)


from baseline_correction_functions import BaselineCorrector
from libs_generic_functions import LibsDataLoader, LibsDataPreprocessor

In [3]:
dataloader = LibsDataLoader(data_directory='data')
preprocessor = LibsDataPreprocessor()
corrector = BaselineCorrector()

In [4]:
df = dataloader.load_all_measurements()

MEMORY-EFFICIENT LOADING OF ALL INDIVIDUAL MEASUREMENTS
First pass: Counting measurements...
Processing file 1/316: 2024-11-05T11-26-41_nr-001_B1_tread_aided_10Hz_280A.h5
Processing file 11/316: 2024-11-05T11-51-23_nr-011_B4_innerliner_aided_10Hz_280A.h5
Processing file 21/316: 2024-11-05T12-02-34_nr-021_B7_sidewall_aided_10Hz_280A.h5
Processing file 31/316: 2024-11-05T12-13-30_nr-031_B22_tread_aided_10Hz_280A.h5
Processing file 41/316: 2024-11-05T12-25-42_nr-041_B33_innerliner_aided_10Hz_280A.h5
Processing file 51/316: 2024-11-05T12-36-31_nr-051_B28_sidewall_aided_10Hz_280A.h5
Processing file 61/316: 2024-11-05T12-47-29_nr-061_B15_tread_aided_10Hz_280A.h5
  Counted 10,000 measurements...
Processing file 71/316: 2024-11-05T12-58-45_nr-071_B30_innerliner_aided_10Hz_280A.h5
Processing file 81/316: 2024-11-05T13-17-11_nr-081_B35_innerliner_aided_10Hz_280A.h5
Processing file 91/316: 2024-11-05T14-05-16_nr-091_B26_sidewall_aided_10Hz_280A.h5
Processing file 101/316: 2024-11-05T14-16-52_nr-1

In [6]:
non_feature_cols = ['tire_number', 'origin', 'measurement_id']
print(f"Non-feature columns: {non_feature_cols}")

# Get numeric columns (wavelength data)
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
wavelength_columns = [col for col in numeric_columns if col not in non_feature_cols]

Non-feature columns: ['tire_number', 'origin', 'measurement_id']


In [8]:
X = df[wavelength_columns].values
y = df['origin'].values

# Encode target labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Dataset shape: {X.shape}")
print(f"Classes: {label_encoder.classes_}")
print(f"Class distribution: {np.bincount(y_encoded)}")

X_corrected = corrector.apply_correction_batch(X)

# 2. Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_corrected)

Dataset shape: (51453, 8188)
Classes: ['innerliner' 'sidewall' 'tread']
Class distribution: [17260 16257 17936]


Applying HYBRID baseline correction to 51453 spectra in batches of 100...
  Processing batch 1/515 (spectra 1-100)
  Processing batch 2/515 (spectra 101-200)
  Processing batch 3/515 (spectra 201-300)
  Processing batch 4/515 (spectra 301-400)
  Processing batch 5/515 (spectra 401-500)
  Processing batch 6/515 (spectra 501-600)
  Processing batch 7/515 (spectra 601-700)
  Processing batch 8/515 (spectra 701-800)
  Processing batch 9/515 (spectra 801-900)
  Processing batch 10/515 (spectra 901-1000)
  Processing batch 11/515 (spectra 1001-1100)
  Processing batch 12/515 (spectra 1101-1200)
  Processing batch 13/515 (spectra 1201-1300)
  Processing batch 14/515 (spectra 1301-1400)
  Processing batch 15/515 (spectra 1401-1500)
  Processing batch 16/515 (spectra 1501-1600)
  Processing batch 17/515 (spectra 1601-1700)
  Processing batch 18/515 (spectra 1701-1800)
  Processing batch 19/515 (spectra 1801-1900)
  Processing batch 20/515 (spectra 1901-2000)
  Processing batch 21/515 (spectra 2

In [ ]:
#replace the original data with the processed data
df.head()
df[wavelength_columns] = X_scaled
df.head()



,200.000,200.098,200.195,200.293,200.391,200.489,200.586,200.684,200.782,200.879,...,999.414,999.511,999.609,999.707,999.805,999.902,1000.000,origin,tire_number,measurement_id
0,-0.042140,-0.098582,0.161465,0.092517,0.216881,0.024452,-0.180463,0.248127,-0.196329,-0.237015,...,-0.699809,-0.696985,-0.660819,-0.919420,-0.754962,-0.755579,-0.707699,tread,1,0
1,-0.534830,-0.537674,-0.359261,-1.038418,-0.530579,-0.566161,-0.135357,-0.504000,-0.271974,-0.323830,...,-0.337096,-0.563596,-0.421811,-0.332108,-0.348489,-0.378949,-0.456849,tread,1,1
2,-0.912142,-0.668835,-0.293863,-0.905609,-0.546995,-0.582210,-1.225461,-1.251652,-1.056249,-0.826600,...,-0.226553,-0.284314,-0.348104,-0.364682,-0.467611,-0.255852,-0.441867,tread,1,2
3,-0.948000,-0.827213,-1.091950,-1.033830,-0.778049,-1.011812,-0.770409,-1.265328,-0.567896,-0.391490,...,-0.643413,-0.660539,-0.609633,-0.409939,-0.584255,-0.502976,-0.551347,tread,1,3
4,-0.516686,-0.871476,-0.813389,-0.732897,-0.809114,-0.724975,-0.117344,-0.043280,-0.526647,-0.002335,...,0.281129,0.363627,0.296198,0.411464,0.372758,0.318723,0.328807,tread,1,4


In [11]:
processed_data_path = 'data/csv/processed_data.csv'
df.to_csv(processed_data_path, index=False)

In [3]:
processed_data_path = 'data/csv/processed_data.csv'
import pandas as pd
df = pd.read_csv(processed_data_path)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51453 entries, 0 to 51452
Columns: 8191 entries, 200.000 to measurement_id
dtypes: float64(8188), int64(2), object(1)
memory usage: 3.1+ GB
